# 使用新模板创建和评估因子

本 Notebook 演示：创建 `factor_analyse/factor_mining/` 下的新模板因子、修改公式、执行评估、读取保存结果和生成 HTML 报告。

请先运行创建单元格，再编辑生成的 `.py` 文件，最后运行评估单元格。

## 1. 初始化路径和环境

In [1]:
# 开发期自动重载仓库内代码改动；本单元格可以重复执行
from pathlib import Path
import os
import sys

ipython = get_ipython()
autoreload_module = "IPython.extensions.autoreload"
if autoreload_module not in ipython.extension_manager.loaded:
    ipython.run_line_magic("load_ext", "autoreload")
ipython.run_line_magic("autoreload", "2")

def find_repo_root(start: Path) -> Path:
    candidates = [start.resolve(), *start.resolve().parents]
    injected = os.environ.get("FACTOR_COMMON_REPO_ROOT")
    if injected:
        candidates.insert(0, Path(injected).expanduser().resolve())
    candidates.append(Path("/Users/dmiwu/work/PythonProject/cryptoFactorAnalyze"))
    for candidate in candidates:
        if (candidate / "factor_common").is_dir() and (candidate / "data").is_dir():
            return candidate
    raise RuntimeError("无法定位 cryptoFactorAnalyze 仓库根目录")

ROOT = find_repo_root(Path.cwd())
# 强制仓库源码排在已安装包之前；重复运行时也保持在第一位
repo_path = str(ROOT)
sys.path[:] = [item for item in sys.path if item != repo_path]
sys.path.insert(0, repo_path)
# 清除当前 kernel 中此前从其他路径导入的 factor_common 模块
for module_name in tuple(sys.modules):
    if module_name == "factor_common" or module_name.startswith("factor_common."):
        del sys.modules[module_name]

print("repo root:", ROOT)


repo root: /Users/dmiwu/work/PythonProject/cryptoFactorAnalyze


In [2]:
from factor_common import FactorManager

# 直接给出因子文件的完整路径（无需在任何注册表中登记）
factor_path = Path("/Users/dmiwu/work/PythonProject/cryptoFactorAnalyze/factor_analyse/factor_mining/Idio_Vol_Factor.py")
FACTOR_NAME = factor_path.stem
H5_PATH = ROOT / "data" / "crypto_quant.h5"

manager = FactorManager(
    h5_path=H5_PATH,
    base_dir=ROOT / "data" / "factor_results",
    reports_dir=ROOT / "reports",
)

print("H5:", manager.h5_path)
print("factor file:", factor_path)


H5: /Users/dmiwu/work/PythonProject/cryptoFactorAnalyze/data/crypto_quant.h5
factor file: /Users/dmiwu/work/PythonProject/cryptoFactorAnalyze/factor_analyse/factor_mining/Idio_Vol_Factor.py


## 2. 创建模板因子

该单元格不会覆盖已有文件。第一次运行会创建 `my_factor.py`；如果文件已经存在，则直接使用它。

In [3]:
if not factor_path.is_file():
    raise FileNotFoundError(f"未找到因子文件: {factor_path}")
print("当前使用的因子文件：", factor_path)
# print(factor_path.read_text(encoding="utf-8"))


当前使用的因子文件： /Users/dmiwu/work/PythonProject/cryptoFactorAnalyze/factor_analyse/factor_mining/Idio_Vol_Factor.py


## 3. 编辑因子公式

打开并编辑下面的文件：

```text
factor_analyse/factor_mining/my_factor.py
```

修改 `SETTING["data_needed"]`、`SETTING["params"]` 和 `calc_factor()`。因子值只能使用当前及历史数据；不要使用负向 `shift`、居中 rolling 或反向填充。编辑完成后回到本 Notebook 继续运行。

## 4. 执行因子评估

In [4]:
# 每次评估前显式重载公共模块，避免 QuantTest kernel 持有旧的 manager/storage 类
import importlib
import inspect
import pandas as pd
import factor_common.storage as storage_module
import factor_common.data_provider as data_provider_module
import factor_common.metrics as metrics_module
import factor_common.manager as manager_module
importlib.reload(storage_module)
importlib.reload(data_provider_module)
importlib.reload(metrics_module)
importlib.reload(manager_module)
FactorManager = manager_module.FactorManager
manager_source = inspect.getsource(FactorManager.evaluate)
print("manager module:", Path(manager_module.__file__).resolve())
if "values = values.where(eligible" not in manager_source:
    raise RuntimeError("当前 kernel 仍加载了旧版 FactorManager，请重启 kernel")

# 每次评估前重建 manager，确保使用当前代码和最新缓存校验逻辑
manager = FactorManager(
    h5_path=H5_PATH,
    base_dir=ROOT / "data" / "factor_results",
    reports_dir=ROOT / "reports",
)

# 回测结束后还需要 1 + rebalance_days 天的开盘价完成持仓退出。
# 若请求日期超过当前 K 线覆盖范围，自动收敛到能够完整结算的最后信号日。
requested_end = pd.Timestamp("2026-09-01")
rebalance_days = 3
_, market_end = manager.dp.get_time_range()
safe_end = market_end - pd.Timedelta(days=1 + rebalance_days)
evaluation_end = min(requested_end, safe_end)
if evaluation_end < requested_end:
    print(
        f"请求结束日 {requested_end.date()} 需要未来平仓价格；"
        f"已自动调整为 {evaluation_end.date()}（K线截至 {market_end.date()}）"
    )

result = manager.evaluate(
    factor_path,
    params={
        "start": "2024-01-01",
        "end": evaluation_end.date().isoformat(),
        "rebalance_days": rebalance_days,
        "n_groups": 5,
        "fee_rate": 0.0005,
        "slippage": 0.001,
        "include_funding": True,
    },
    plot=False,
)

print("status:", result["status"])
print("run_id:", result["run_id"])
print("evaluation_id:", result["evaluation_id"])
print("factor value shape:", result["factor_value"].shape)
print("report:", result["paths"]["report_path"])


manager module: /Users/dmiwu/work/PythonProject/cryptoFactorAnalyze/factor_common/manager.py
请求结束日 2026-09-01 需要未来平仓价格；已自动调整为 2026-08-31（K线截至 2026-09-04）


/Users/dmiwu/work/PythonProject/cryptoFactorAnalyze/factor_analyse/factor_mining/Idio_Vol_Factor.py:100: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  ret = close.pct_change(periods=1)
/Users/dmiwu/work/PythonProject/cryptoFactorAnalyze/factor_analyse/factor_mining/Idio_Vol_Factor.py:100: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  ret = close.pct_change(periods=1)
/Users/dmiwu/work/PythonProject/cryptoFactorAnalyze/factor_analyse/factor_mining/Idio_Vol_Factor.py:100: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future versio

status: complete
run_id: a5c6bd0ec55f736d
evaluation_id: None
factor value shape: (974, 157)
report: None


## 5. 查看因子值和 IC

In [5]:
factor_value = result["factor_value"]
display(factor_value.tail())

full_ic = result["factor_performance"]["samples"]["full"]["ic"]
print({
    key: full_ic[key]
    for key in ("ic_mean", "rank_ic_mean", "icir", "t_stat", "n_dates")
})

,1INCHUSDT,2ZUSDT,AAVEUSDT,ADAUSDT,AGIXUSDT,ALGOUSDT,APEUSDT,APTUSDT,ARBUSDT,ARUSDT,...,XLMUSDT,XMRUSDT,XPLUSDT,XRPUSDT,XTZUSDT,ZECUSDT,ZKJUSDT,ZKUSDT,ZROUSDT,币安人生USDT
date,,,,,,,,,,,,,,,,,,,,,
2026-08-27,NaN,NaN,0.265306,0.183673,NaN,-0.102041,NaN,-0.673469,0.428571,NaN,...,-0.061224,0.142857,NaN,-0.510204,NaN,0.673469,NaN,NaN,NaN,0.877551
2026-08-28,NaN,NaN,0.265306,0.183673,NaN,-0.183673,NaN,-0.673469,0.428571,NaN,...,-0.020408,0.142857,NaN,-0.510204,NaN,0.673469,NaN,NaN,NaN,0.877551
2026-08-29,NaN,NaN,0.265306,0.183673,NaN,-0.142857,NaN,-0.632653,0.387755,NaN,...,-0.755102,0.102041,NaN,-0.469388,NaN,0.673469,NaN,NaN,NaN,0.877551
2026-08-30,NaN,NaN,0.265306,0.142857,NaN,-0.183673,NaN,-0.632653,0.428571,NaN,...,-0.795918,0.102041,NaN,-0.469388,NaN,0.714286,NaN,NaN,NaN,0.877551
2026-08-31,NaN,NaN,0.265306,0.102041,NaN,-0.183673,NaN,-0.591837,0.836735,NaN,...,-0.836735,0.142857,NaN,-0.428571,NaN,0.673469,NaN,NaN,NaN,0.877551


{'ic_mean': -0.03237825298184274, 'rank_ic_mean': -0.06935436984278451, 'icir': -0.15072002817041913, 't_stat': -4.701401989366178, 'n_dates': 973}


## 6. 读取已保存因子值并生成报告

In [6]:
print("report source status:", result["status"])
print("report source run_id:", result["run_id"])
if result["status"] != "complete":
    print("回测存在未认证区间，报告仍会生成。中止原因：")
    for scenario_name, scenario in result["factor_result"]["scenarios"].items():
        diagnostics = scenario["diagnostics"]
        print(
            f"  {scenario_name}: {diagnostics['halt_reason']} "
            f"on {diagnostics['halt_date']} ({diagnostics['halt_detail']})"
        )

reloaded_value = manager.get_value(FACTOR_NAME, run_id=result["run_id"])
print("reload equals computed value:", reloaded_value.equals(factor_value))

saved_result = result

report_info = manager.plot_result(
    saved_result,
    output_path=ROOT / "reports" / f"{FACTOR_NAME}_report.html",
)
print("reloaded report:", report_info["output_path"])

report source status: complete
report source run_id: a5c6bd0ec55f736d
reload equals computed value: True
reloaded report: /Users/dmiwu/work/PythonProject/cryptoFactorAnalyze/reports/Idio_Vol_Factor_report.html


## 7. 查看未来函数检查和资金覆盖诊断

In [7]:
validation = result["diagnostics"]["validation"]
print("static scan:", validation["static_scan"]["status"])
print("cutoff replay:", validation["cutoff"]["status"])
print("cutoff diffs:", [
    item["max_abs_diff"]
    for item in validation["cutoff"].get("cutoffs", [])
])
print("funding coverage:", result["diagnostics"]["coverage"]["funding"]["status_counts"])

static scan: clean
cutoff replay: verified
cutoff diffs: [0.0, 0.0]
funding coverage: {'complete': 20327}
